# Data Visualization Notebook

Create visualizations to understand patterns and trends

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

# Seaborn is optional - works without it
try:
    import seaborn as sns
    sns.set_style('whitegrid')
    SEABORN_AVAILABLE = True
    print('seaborn loaded')
except ImportError:
    SEABORN_AVAILABLE = False
    print('seaborn not installed - using matplotlib fallbacks')

plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 10
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

print('Libraries imported successfully!')

## 1. Load Cleaned Data

In [ ]:
# Load processed data
df = pd.read_csv('../dataset/processed_data/cleaned_data.csv')
df['Date'] = pd.to_datetime(df['Date'])

print(f'Data Shape: {df.shape}')
print(f'Columns: {df.columns.tolist()}')
print(f'Date Range: {df["Date"].min()} to {df["Date"].max()}')

## 2. Price Trend Over Time (Line Chart)

In [ ]:
# Create output directory
output_dir = '../outputs/graphs'
os.makedirs(output_dir, exist_ok=True)

# Line chart showing price trend by commodity
fig, ax = plt.subplots(figsize=(14, 6))

# Plot price trend for each commodity
for commodity_id in df['Commodity'].unique():
    data = df[df['Commodity'] == commodity_id].sort_values('Date')
    ax.plot(data['Date'], data['Modal Price'], marker='o', 
            label=f'Commodity {int(commodity_id)}', linewidth=2, markersize=6)

ax.set_xlabel('Date', fontsize=12, fontweight='bold')
ax.set_ylabel('Modal Price ($)', fontsize=12, fontweight='bold')
ax.set_title('Price Trend Over Time by Commodity', fontsize=14, fontweight='bold')
ax.legend(fontsize=10, loc='best')
ax.grid(True, alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()

# Save chart
filepath = f'{output_dir}/price_trend.png'
plt.savefig(filepath, dpi=300, bbox_inches='tight')
print(f'Saved: price_trend.png')
plt.show()

## 3. Average Price by Commodity (Bar Chart)

In [ ]:
# Bar chart showing average price per commodity
fig, ax = plt.subplots(figsize=(10, 6))

# Calculate average price by commodity
avg_price = df.groupby('Commodity')['Modal Price'].mean().sort_values(ascending=False)

# Create color palette using matplotlib
cmap = plt.cm.viridis
colors = [cmap(i / len(avg_price)) for i in range(len(avg_price))]

# Plot bars
bars = ax.bar(range(len(avg_price)), avg_price.values, color=colors, edgecolor='black', linewidth=1.5)
ax.set_xlabel('Commodity ID', fontsize=12, fontweight='bold')
ax.set_ylabel('Average Modal Price ($)', fontsize=12, fontweight='bold')
ax.set_title('Average Price by Commodity', fontsize=14, fontweight='bold')
ax.set_xticks(range(len(avg_price)))
ax.set_xticklabels([f'C{int(i)}' for i in avg_price.index])
ax.grid(True, alpha=0.3, axis='y')

# Add value labels on bars
for i, v in enumerate(avg_price.values):
    ax.text(i, v + 50, f'${v:.0f}', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()

# Save chart
filepath = f'{output_dir}/avg_price_by_commodity.png'
plt.savefig(filepath, dpi=300, bbox_inches='tight')
print(f'Saved: avg_price_by_commodity.png')
plt.show()

## 4. Price Distribution by Commodity (Box Plot)

In [ ]:
# Box plot showing price distribution per commodity
fig, ax = plt.subplots(figsize=(10, 6))

# Create data copy for plotting
df_plot = df.copy()
df_plot['Commodity_Label'] = 'Commodity ' + df_plot['Commodity'].astype(int).astype(str)

# Create box plot using matplotlib
commodities = sorted(df_plot['Commodity_Label'].unique())
data_by_commodity = [df_plot[df_plot['Commodity_Label'] == c]['Modal Price'].values for c in commodities]
bp = ax.boxplot(data_by_commodity, labels=commodities, patch_artist=True)

# Color the boxes
colors = plt.cm.Set2(np.linspace(0, 1, len(commodities)))
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)

ax.set_xlabel('Commodity', fontsize=12, fontweight='bold')
ax.set_ylabel('Modal Price ($)', fontsize=12, fontweight='bold')
ax.set_title('Price Distribution by Commodity', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()

# Save chart
filepath = f'{output_dir}/price_distribution.png'
plt.savefig(filepath, dpi=300, bbox_inches='tight')
print(f'Saved: price_distribution.png')
plt.show()

## 5. Monthly Average Prices - Seasonal Pattern (Heatmap)

In [ ]:
# Create month-commodity pivot for heatmap
df_heat = df.copy()
df_heat['YearMonth'] = df_heat['Date'].dt.to_period('M')
df_heat['Commodity_Label'] = 'C' + df_heat['Commodity'].astype(int).astype(str)

# Calculate monthly averages
monthly_avg = df_heat.groupby(['YearMonth', 'Commodity_Label'])['Modal Price'].mean().unstack()

# Create heatmap using matplotlib
fig, ax = plt.subplots(figsize=(12, 6))

data = monthly_avg.T.values
im = ax.imshow(data, cmap='RdYlGn', aspect='auto')
cbar = plt.colorbar(im, ax=ax)
cbar.set_label('Average Modal Price ($)')

# Set ticks
ax.set_xticks(np.arange(len(monthly_avg.index)))
ax.set_yticks(np.arange(len(monthly_avg.columns)))
ax.set_xticklabels([str(m) for m in monthly_avg.index], rotation=45, ha='right')
ax.set_yticklabels(monthly_avg.columns)

# Add annotations
for i in range(len(monthly_avg.columns)):
    for j in range(len(monthly_avg.index)):
        val = data[i, j]
        if not np.isnan(val):
            ax.text(j, i, f'{val:.0f}', ha='center', va='center', fontsize=8)

ax.set_xlabel('Month', fontsize=12, fontweight='bold')
ax.set_ylabel('Commodity', fontsize=12, fontweight='bold')
ax.set_title('Monthly Average Prices - Seasonal Pattern', fontsize=14, fontweight='bold')
plt.tight_layout()

# Save chart
filepath = f'{output_dir}/seasonal_heatmap.png'
plt.savefig(filepath, dpi=300, bbox_inches='tight')
print(f'Saved: seasonal_heatmap.png')
plt.show()

## 6. Visualization Summary

In [ ]:
# Print summary of visualizations created
print('='*60)
print('VISUALIZATION SUMMARY')
print('='*60)
print(f'\nOutput Directory: {output_dir}')
print(f'\nCharts Created:')

files_created = [
    'price_trend.png',
    'avg_price_by_commodity.png',
    'price_distribution.png',
    'seasonal_heatmap.png'
]

for i, file in enumerate(files_created, 1):
    filepath = f'{output_dir}/{file}'
    if os.path.exists(filepath):
        size = os.path.getsize(filepath) / 1024
        print(f'{i}. {file} ({size:.1f} KB)')

print(f'\nAll visualizations saved successfully!')
print(f'\nKey Insights:')
print(f'  - Commodity {int(df.groupby("Commodity")["Modal Price"].mean().idxmax())}: Highest avg price')
print(f'  - Commodity {int(df.groupby("Commodity")["Modal Price"].mean().idxmin())}: Lowest avg price')
print(f'  - Price range: ${df["Modal Price"].min():.2f} - ${df["Modal Price"].max():.2f}')
print(f'\nNext: Run src/model.py for ML training')